### Notebook for extracting and analysing LFP data from Neuropixels recordings

In [ ]:
# Imports
import os
import yaml

import numpy as np
import matplotlib.pyplot as plt
import spikeinterface.core as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp
import spikeinterface.sorters as ss

from scipy.signal import spectrogram
from probeinterface import get_probe, Probe
from probeinterface.plotting import plot_probe
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

In [ ]:
load_dotenv()
repo_root = Path(find_dotenv()).parent

raw_data_dir = Path(os.getenv("RAW_DATA_DIR"))
data_fname = Path(os.getenv("TEST_DATA_FNAME"))
derivatives_dir = Path(os.getenv("DERIVATIVES_DIR"))
full_data_path = os.path.join(raw_data_dir, data_fname)
config_path = os.path.join(repo_root, "config/preprocessing.yml")
lfp_output_dir = os.path.join(derivatives_dir, "lfp")

# Load preprocessing config
with open(Path(config_path)) as f:
    config = yaml.safe_load(f)
    
sample_rate = config['recording']['sample_rate']
n_channels = config['recording']['n_channels']
dtype = config['recording']['dtype']
freq_min = config['lfp']['bandpass_filter']['freq_min']
freq_max = config['lfp']['bandpass_filter']['freq_max']
reference = config['lfp']['common_reference']['reference']
operator = config['lfp']['common_reference']['operator']
resample_rate = config['lfp']['resample_rate']

In [ ]:
recording = se.read_binary(full_data_path, sample_rate, dtype, n_channels)

In [ ]:
probe_full = get_probe("imec", "NP2013")

# Extract shank 0 contacts only
shank_ids = probe_full.shank_ids
shank0_mask = probe_full.shank_ids == "0"
shank0_pos = probe_full.contact_positions[shank0_mask]
sorted_idx = np.argsort(shank0_pos[:, 1])  # ascending = bottom first
bottom_384_pos = shank0_pos[sorted_idx[:384]]

probe = Probe(ndim=2, si_units="um")
probe.set_contacts(
    positions=bottom_384_pos,
    shapes="circle",
    shape_params={"radius": 7.5}
)
probe.set_device_channel_indices(np.arange(384))
probe.create_auto_shape(probe_type="tip")

recording_with_probe = recording.set_probe(probe)

In [ ]:
lfp = sp.bandpass_filter(recording_with_probe, freq_min=0.5, freq_max=300)
lfp = sp.common_reference(lfp, reference="global", operator="median")
lfp = sp.resample(lfp, resample_rate=1250)

In [ ]:
fs = lfp.get_sampling_frequency()  # 1250
chunk_size = int(60 * fs)  # 5 minutes at a time
n_total = lfp.get_num_samples()
channel_idx = 6

all_freqs, all_times, all_Sxx = [], [], []

for (loop_no, start) in enumerate(range(0, chunk_size*5, chunk_size)):
    end = min(start + chunk_size, n_total)
    chunk = lfp.get_traces(start_frame=start, end_frame=end)
    lfp_trace = chunk[:, channel_idx]
    freqs, times, Sxx = spectrogram(
        lfp_trace,
        fs=1250,
        window='hann',
        nperseg=1250 * 4,    # 4-second window
        noverlap=1250 * 3,   # 75% overlap → 1-second steps
    )
    all_freqs.append(freqs)
    all_times.append(times)
    all_Sxx.append(Sxx)
    print(f'Finished chunk {loop_no}')

In [ ]:
freqs = all_freqs[0]  # same for all chunks

# Offset times by chunk start, then concatenate
times = np.concatenate([
    all_times[i] + (i * 60)  # offset by chunk number * chunk duration in seconds
    for i in range(len(all_times))
])

Sxx = np.concatenate(all_Sxx, axis=1)  # concatenate along time axis

print(f"freqs shape: {freqs.shape}")
print(f"times shape: {times.shape}")
print(f"Sxx shape:   {Sxx.shape}")  # should be (n_freqs, n_times)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(18, 5))

ax.pcolormesh(
    times,           # convert to hours
    freqs,
    10 * np.log10(Sxx),     # convert to dB
    cmap="inferno",
    shading="auto",
    vmin=-10,
    vmax=40
)

ax.set_ylim(0, 30)          # focus on sleep-relevant bands
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title(f"LFP spectrogram — channel {channel_idx}")

plt.colorbar(ax.collections[0], ax=ax, label="Power (dB)")
plt.tight_layout()
plt.show()